In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('CIC-DDoS-2019_training.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50063112 entries, 0 to 50063111
Data columns (total 84 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Flow ID            object 
 1   Src IP             object 
 2   Src Port           int64  
 3   Dst IP             object 
 4   Dst Port           int64  
 5   Protocol           int64  
 6   Timestamp          object 
 7   Flow Duration      int64  
 8   Tot Fwd Pkts       int64  
 9   Tot Bwd Pkts       int64  
 10  TotLen Fwd Pkts    float64
 11  TotLen Bwd Pkts    float64
 12  Fwd Pkt Len Max    float64
 13  Fwd Pkt Len Min    float64
 14  Fwd Pkt Len Mean   float64
 15  Fwd Pkt Len Std    float64
 16  Bwd Pkt Len Max    float64
 17  Bwd Pkt Len Min    float64
 18  Bwd Pkt Len Mean   float64
 19  Bwd Pkt Len Std    float64
 20  Flow Byts/s        float64
 21  Flow Pkts/s        float64
 22  Flow IAT Mean      float64
 23  Flow IAT Std       float64
 24  Flow IAT Max       float64
 25  Flow IAT Min    

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
global_dups = df.duplicated().sum()

label_dups = df.groupby("Label").apply(
    lambda x: x.duplicated().sum()
).sum()

print(global_dups, label_dups)


0 0


C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_12604\3639644098.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  label_dups = df.groupby("Label").apply(


In [6]:
print(df["Label"].isna().sum())

0


In [7]:
# Use dropna=False to include the rows with missing labels
df.groupby("Label", dropna=False).apply(lambda x: x.duplicated().sum())

C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_12604\2161749747.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Label", dropna=False).apply(lambda x: x.duplicated().sum())


Label
BENIGN           0
DrDoS_DNS        0
DrDoS_LDAP       0
DrDoS_MSSQL      0
DrDoS_NTP        0
DrDoS_NetBIOS    0
DrDoS_SNMP       0
DrDoS_SSDP       0
DrDoS_UDP        0
Syn              0
TFTP             0
UDP-lag          0
WebDDoS          0
dtype: int64

In [8]:
df.shape

(50063112, 84)

In [9]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 249024


In [10]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 249024


In [11]:
# This will show only the columns that have at least one NaN
print(df.columns[df.isna().any()].tolist())

['Flow Byts/s']


In [12]:
df['Flow Byts/s'].isna().sum()

np.int64(249024)

In [13]:
# 1. Filter the dataframe for only rows where Flow Bytes/s is NaN
nan_data = df[df['Flow Byts/s'].isna()]

# 2. Count the labels within that subset
distribution = nan_data['Label'].value_counts()

print("Distribution of NaNs per Label:")
print(distribution)

# 3. Optional: See the percentage of each label that is "broken"
total_per_label = df['Label'].value_counts()
nan_percentage = (distribution / total_per_label) * 100
print("\nPercentage of each Label that has NaNs:")
print(nan_percentage.dropna())

Distribution of NaNs per Label:
Label
Syn              202274
UDP-lag           36132
TFTP              10472
BENIGN              113
DrDoS_DNS             9
DrDoS_SNMP            7
DrDoS_NetBIOS         6
DrDoS_MSSQL           3
DrDoS_NTP             3
DrDoS_LDAP            2
DrDoS_UDP             2
DrDoS_SSDP            1
Name: count, dtype: int64

Percentage of each Label that has NaNs:
Label
BENIGN            0.198723
DrDoS_DNS         0.000177
DrDoS_LDAP        0.000092
DrDoS_MSSQL       0.000066
DrDoS_NTP         0.000249
DrDoS_NetBIOS     0.000147
DrDoS_SNMP        0.000136
DrDoS_SSDP        0.000038
DrDoS_UDP         0.000064
Syn              12.783632
TFTP              0.052145
UDP-lag           9.859712
Name: count, dtype: float64


In [14]:
df['Label'].value_counts()

Label
TFTP             20082580
DrDoS_SNMP        5159870
DrDoS_DNS         5071011
DrDoS_MSSQL       4522492
DrDoS_NetBIOS     4093279
DrDoS_UDP         3134645
DrDoS_SSDP        2610611
DrDoS_LDAP        2179930
Syn               1582289
DrDoS_NTP         1202642
UDP-lag            366461
BENIGN              56863
WebDDoS               439
Name: count, dtype: int64

In [15]:
# 1. Select only the columns that contain text/objects
string_columns = df.select_dtypes(include=['object']).columns

# 2. Check if the stripped string is empty
# .str.strip() removes spaces; .eq('') checks if it is then empty
blanks_mask = df[string_columns].apply(lambda x: x.str.strip().eq('')).any(axis=1)

total_blank_rows = blanks_mask.sum()
print(f"Total rows with at least one blank entry: {total_blank_rows}")

Total rows with at least one blank entry: 0


In [16]:
import numpy as np

# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Total number of infinite values
total_inf = np.isinf(numeric_df).values.sum()

print(f"Total infinite entries: {total_inf}")

Total infinite entries: 2477448


In [17]:
# Check if any value in a row is infinite
inf_rows = np.isinf(numeric_df).any(axis=1).sum()

print(f"Total rows with at least one infinite value: {inf_rows}")

Total rows with at least one infinite value: 1363236


In [18]:
import numpy as np

# Select only numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Count infinite values (positive and negative) per column
inf_counts = np.isinf(df[numeric_cols]).sum()

# Display only columns that have at least one infinite value
print("Infinite values per column:")
print(inf_counts[inf_counts > 0])

Infinite values per column:
Flow Byts/s    1114212
Flow Pkts/s    1363236
dtype: int64


In [19]:
# 1. Create a boolean mask for any row that has at least one infinite value
inf_rows_mask = np.isinf(df[numeric_cols]).any(axis=1)

# 2. Filter the dataframe using that mask and count the Labels
inf_label_dist = df[inf_rows_mask]['Label'].value_counts()

print("\nDistribution of Infinite entries per Label:")
print(inf_label_dist)


Distribution of Infinite entries per Label:
Label
TFTP             566609
Syn              202306
DrDoS_DNS        162346
DrDoS_NetBIOS    129833
DrDoS_MSSQL      126446
DrDoS_SSDP        42042
DrDoS_UDP         40643
DrDoS_LDAP        38630
UDP-lag           36382
DrDoS_SNMP        10609
DrDoS_NTP          6952
BENIGN              438
Name: count, dtype: int64


In [20]:
total_inf_cells = np.isinf(df[numeric_cols]).values.sum()
total_rows_with_inf = inf_rows_mask.sum()

print(f"\nTotal infinite cells in dataset: {total_inf_cells}")
print(f"Total rows affected by infinity: {total_rows_with_inf}")


Total infinite cells in dataset: 2477448
Total rows affected by infinity: 1363236


In [21]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 249024


In [22]:
# This will show only the columns that have at least one NaN
print(df.columns[df.isna().any()].tolist())

['Flow Byts/s']


In [23]:
# 1. Filter the dataframe for only rows where Flow Bytes/s is NaN
nan_data = df[df['Flow Byts/s'].isna()]

# 2. Count the labels within that subset
distribution = nan_data['Label'].value_counts()

print("Distribution of NaNs per Label:")
print(distribution)

# 3. Optional: See the percentage of each label that is "broken"
total_per_label = df['Label'].value_counts()
nan_percentage = (distribution / total_per_label) * 100
print("\nPercentage of each Label that has NaNs:")
print(nan_percentage.dropna())

Distribution of NaNs per Label:
Label
Syn              202274
UDP-lag           36132
TFTP              10472
BENIGN              113
DrDoS_DNS             9
DrDoS_SNMP            7
DrDoS_NetBIOS         6
DrDoS_MSSQL           3
DrDoS_NTP             3
DrDoS_LDAP            2
DrDoS_UDP             2
DrDoS_SSDP            1
Name: count, dtype: int64

Percentage of each Label that has NaNs:
Label
BENIGN            0.198723
DrDoS_DNS         0.000177
DrDoS_LDAP        0.000092
DrDoS_MSSQL       0.000066
DrDoS_NTP         0.000249
DrDoS_NetBIOS     0.000147
DrDoS_SNMP        0.000136
DrDoS_SSDP        0.000038
DrDoS_UDP         0.000064
Syn              12.783632
TFTP              0.052145
UDP-lag           9.859712
Name: count, dtype: float64


In [24]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [25]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 2726472


In [26]:
df.describe()

,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,...,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07,5.006311e+07
mean,3.266417e+04,3.283233e+04,1.653778e+01,1.123535e+06,4.986730e+00,2.201056e-02,2.394542e+03,5.960831e+00,6.793276e+02,6.737759e+02,...,3.269073e+00,-4.483264e+07,6.132369e+02,2.211365e+02,9.554598e+02,4.910094e+02,7.828654e+04,2.050911e+04,1.058493e+05,5.875499e+04
std,2.746077e+04,1.890640e+04,2.208164e+00,5.507000e+06,2.444261e+02,1.496926e+00,6.685970e+03,3.990884e+03,4.563536e+02,4.603182e+02,...,1.404742e+01,2.136268e+08,6.230542e+04,2.435963e+04,8.077155e+04,5.870855e+04,1.274560e+06,4.202331e+05,1.728602e+06,1.020585e+06
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-1.408238e+09,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.440000e+02,1.648700e+04,1.700000e+01,1.000000e+00,2.000000e+00,0.000000e+00,1.032000e+03,0.000000e+00,4.090000e+02,4.010000e+02,...,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,3.828700e+04,3.284000e+04,1.700000e+01,2.000000e+00,2.000000e+00,0.000000e+00,1.810000e+03,0.000000e+00,5.160000e+02,5.160000e+02,...,1.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,6.247800e+04,4.921500e+04,1.700000e+01,1.081140e+05,4.000000e+00,0.000000e+00,2.928000e+03,0.000000e+00,7.200000e+02,7.190000e+02,...,3.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553400e+04,6.553500e+04,1.700000e+01,1.200000e+08,1.001480e+05,4.602000e+03,1.526642e+07,1.099376e+07,3.212000e+04,2.021000e+03,...,5.043000e+03,1.480000e+03,6.151289e+07,4.868047e+07,7.286843e+07,6.151289e+07,1.192194e+08,6.600292e+07,1.192194e+08,1.192194e+08


In [27]:
df.shape

(50063112, 84)

In [28]:
# Save the cleaned DataFrame back to your file
df.to_csv('CIC-DDoS-2019_training.csv', index=False)

In [29]:
pd.set_option('display.max_rows', None)

# 2. Run your zero-count check again
# Replace 'df' with the name of your dataframe (e.g., X_normalized or data)
zero_counts = (df == 0).sum()

# 3. Print the result
print(zero_counts)

Flow ID                     0
Src IP                      0
Src Port                 2546
Dst IP                      0
Dst Port                 2546
Protocol                 2546
Timestamp                   0
Flow Duration         1363236
Tot Fwd Pkts                0
Tot Bwd Pkts         49726420
TotLen Fwd Pkts       2080711
TotLen Bwd Pkts      50027873
Fwd Pkt Len Max       2080711
Fwd Pkt Len Min       2099903
Fwd Pkt Len Mean      2080711
Fwd Pkt Len Std      46702116
Bwd Pkt Len Max      50027873
Bwd Pkt Len Min      50040643
Bwd Pkt Len Mean     50027873
Bwd Pkt Len Std      50050239
Flow Byts/s           1830959
Flow Pkts/s                 0
Flow IAT Mean         1363236
Flow IAT Std         34945190
Flow IAT Max          1363236
Flow IAT Min          4102703
Fwd IAT Tot           1407944
Fwd IAT Mean          1407944
Fwd IAT Std          35174113
Fwd IAT Max           1407944
Fwd IAT Min           4092608
Bwd IAT Tot          49764683
Bwd IAT Mean         49764683
Bwd IAT St

In [30]:
pd.set_option('display.max_rows', None)

# 2. Calculate the percentage of zeros
# (df_test_new == 0).mean() gives the proportion, * 100 gives the %
zero_percentage = (df == 0).mean() * 100

# 3. Print the result (sorted descending so you see the "emptiest" columns first)
print(zero_percentage)

Flow ID                0.000000
Src IP                 0.000000
Src Port               0.005086
Dst IP                 0.000000
Dst Port               0.005086
Protocol               0.005086
Timestamp              0.000000
Flow Duration          2.723035
Tot Fwd Pkts           0.000000
Tot Bwd Pkts          99.327465
TotLen Fwd Pkts        4.156176
TotLen Bwd Pkts       99.929611
Fwd Pkt Len Max        4.156176
Fwd Pkt Len Min        4.194512
Fwd Pkt Len Mean       4.156176
Fwd Pkt Len Std       93.286482
Bwd Pkt Len Max       99.929611
Bwd Pkt Len Min       99.955119
Bwd Pkt Len Mean      99.929611
Bwd Pkt Len Std       99.974286
Flow Byts/s            3.657302
Flow Pkts/s            0.000000
Flow IAT Mean          2.723035
Flow IAT Std          69.802273
Flow IAT Max           2.723035
Flow IAT Min           8.195062
Fwd IAT Tot            2.812338
Fwd IAT Mean           2.812338
Fwd IAT Std           70.259542
Fwd IAT Max            2.812338
Fwd IAT Min            8.174897
Bwd IAT 